# Run FIM Itemset Comparison

Runs an itemset-only benchmark for CPU/GPU bitset and external FIM baselines, writing outputs into `data/`.


In [ ]:
from pathlib import Path
import sys

current_dir = Path.cwd()
if not (current_dir / 'fim_comparison.py').exists():
    candidate = (Path.cwd() / 'notebooks' / 'profiling' / 'fim_compare').resolve()
    if (candidate / 'fim_comparison.py').exists():
        current_dir = candidate

if str(current_dir) not in sys.path:
    sys.path.insert(0, str(current_dir))

from fim_comparison import run_benchmark, summarize_records

print('current_dir:', current_dir)


In [ ]:
# Configuration
REPEAT_FACTORS = [1]
RUNS = 5
WARMUP_RUNS = 1
ALGORITHMS = [
    'bitset_fim_cpu',
    'bitset_fim_gpu',
    'pyfim_apriori',
    'pyfim_eclat',
    'mlxtend_apriori',
    'spmf_fpgrowth',
    'spmf_eclat',
]
MIN_SUPPORT_COUNT = 80
MIN_SUPPORT_RATIO = 0.05  # if set, overrides MIN_SUPPORT_COUNT per dataset
MIN_CONFIDENCE = 0.6  # legacy parameter; ignored in the itemset-only benchmark
MAX_LEN = 3
MAX_APYORI_RECORDS = 200000
SPMF_JAR = None
SPMF_FPGROWTH_ALGO = 'FPGrowth_itemsets'
SPMF_ECLAT_ALGO = 'Eclat'
CPP_FIM_CMD = ''
DATASET_PATHS = [
    (current_dir.parent.parent / 'data' / 'telco.csv').resolve(),
    (current_dir.parent.parent / 'data' / 'adult.csv').resolve(),
    (current_dir.parent.parent / 'data' / 'census_income.csv').resolve(),
]
DATASET_SEP = 'auto'
TX_COLUMNS = ''  # '' or 'auto' -> automatic; otherwise comma-separated column names
TAG = ''

OUTPUT_DIR = current_dir / 'data'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('REPEAT_FACTORS:', REPEAT_FACTORS)
print('RUNS:', RUNS)
print('WARMUP_RUNS:', WARMUP_RUNS)
print('ALGORITHMS:', ALGORITHMS)
print('DATASET_PATHS:', DATASET_PATHS)
print('OUTPUT_DIR:', OUTPUT_DIR)


In [ ]:
records, output_paths = run_benchmark(
    repeat_factors=[int(x) for x in REPEAT_FACTORS],
    runs=int(RUNS),
    warmup_runs=int(WARMUP_RUNS),
    algorithms=[str(x) for x in ALGORITHMS],
    min_support_count=int(MIN_SUPPORT_COUNT),
    min_support_ratio=None if MIN_SUPPORT_RATIO is None else float(MIN_SUPPORT_RATIO),
    min_confidence=float(MIN_CONFIDENCE),
    max_len=int(MAX_LEN),
    max_apyori_records=int(MAX_APYORI_RECORDS),
    output_dir=OUTPUT_DIR,
    spmf_jar=SPMF_JAR,
    spmf_fpgrowth_algo=SPMF_FPGROWTH_ALGO,
    spmf_eclat_algo=SPMF_ECLAT_ALGO,
    cpp_fim_cmd=CPP_FIM_CMD,
    dataset_paths=DATASET_PATHS,
    dataset_sep=DATASET_SEP,
    tx_columns=None if str(TX_COLUMNS).strip() in {'', 'auto', 'all'} else [x.strip() for x in str(TX_COLUMNS).split(',') if x.strip()],
    tag=str(TAG),
)

print('Saved outputs:')
for key, path in output_paths.items():
    print('-', key, path)


In [ ]:
import pandas as pd

df = pd.DataFrame(records)
summary = summarize_records(records)
df.head(), summary
